# Shared Context Retriever setup — PowerShell

This is the **PowerShell** version, with executable code cells. For executable Bash cells, open
[00 — Shared Context Retriever setup — Bash](00-context-retriever-setup-bash.ipynb).

Run this once before notebooks **03–05**. Select the **.NET (PowerShell)** / Polyglot Notebooks kernel.
Each code cell selects PowerShell with `#!pwsh`. Use PowerShell 7.4+ for UTF-8 native-command redirection.
The commands also work in a PowerShell terminal when you omit that directive.
Start inside this repository (for example, `notebooks/csharp`); the first cell locates the repository root.
Choose one setup notebook; there is no need to provision and seed twice. Resolve failures before running later cells.

This notebook generates the DevUI's travel DataModel, creates or updates a real Context Retriever surface,
creates scoped demo credentials, and imports nine synthetic records. It uses the existing one-shot Docker
Compose jobs; it does not start the DevUI or observability stack. Reuse this surface across the series.

You need Docker with Compose, the repository's .NET SDK, an existing Redis database compatible with your
Context Retriever deployment, and a Context Retriever **admin** endpoint/key. Setup needs no OpenAI or RAM credentials.
Use a dedicated demo database: the seed uses stable keys such as `traveler:traveler-001`, so a different surface
name alone does not isolate records in the same Redis database. Rerunning resets those demo records and updates
the named surface's model. Keep the generated credential bundle between runs.

Follow [the configuration guide](../../samples/RedisIrisDevUI/README.md#provision-the-managed-context-retriever-surface).
In the repository root `.env.local`, configure:

```dotenv
CONTEXT_RETRIEVER_ADMIN_URL="https://cloud.redis.io/context-surfaces"
CONTEXT_RETRIEVER_ADMIN_KEY="cs_admin_..."
CONTEXT_RETRIEVER_REDIS_ADDRESS="your-redis-host:port"
CONTEXT_RETRIEVER_REDIS_USERNAME="default"
CONTEXT_RETRIEVER_REDIS_PASSWORD="..."
CONTEXT_RETRIEVER_REDIS_TLS="true"
CONTEXT_RETRIEVER_SURFACE_NAME="redis-iris-travel-assistant"
CONTEXT_RETRIEVER_CREDENTIAL_MODE="local-users"
REDIS_IRIS_CONTEXT_RETRIEVER_MCP_ENDPOINT="https://your-runtime-endpoint/mcp"
```

The control-plane URL and runtime MCP endpoint are different. Obtain the runtime endpoint for your deployment;
the notebook does not derive it from the surface ID. The setup jobs read `.env.local` directly, independently of
`REDIS_IRIS_ENV_FILE`. For an already configured self-managed deployment, follow [Kubernetes setup](../../docs/kubernetes-samples.md)
and skip the two Docker job steps below. Point notebooks 03–05 at its settings, model artifact, and credential bundle.

Never paste credentials into cells or display configuration/bundle objects. The runtime notebooks load only the
scoped agent identity; administrative operations stay in this setup step.


## Locate the repository root

Run this initialization once. It also works if the shell is already at the repository root.


In [ ]:
#!pwsh
$ErrorActionPreference = 'Stop'
$PSNativeCommandUseErrorActionPreference = $true
if ($PSVersionTable.PSVersion -lt [version]'7.4') {
    throw 'Use PowerShell 7.4 or later for UTF-8 native-command redirection.'
}
while (-not (Test-Path -LiteralPath 'ms-agent-framework-redis.slnx')) {
    $parent = Split-Path -Parent (Get-Location).Path
    if (-not $parent) { throw 'Start inside the repository.' }
    Set-Location -LiteralPath $parent
}


## Build the connector assemblies

This also builds the sample used to generate the model and seed data.


In [ ]:
#!pwsh
dotnet build ms-agent-framework-redis.slnx --configuration Release


## Generate public model and seed artifacts locally

These exports require only the built sample, with no Redis or model credentials. They also provide the model
artifact required by notebooks 03–05 when reusing a Kubernetes fixture. Skip the following two Docker job steps
when your surface and scoped credentials are already provisioned and seeded.


In [ ]:
#!pwsh
New-Item -ItemType Directory -Force -Path artifacts/context-retriever | Out-Null
dotnet samples/RedisIrisDevUI/bin/Release/net8.0/RedisIrisDevUI.dll --print-context-model > artifacts/context-retriever/redis-iris-travel-context-model.json
dotnet samples/RedisIrisDevUI/bin/Release/net8.0/RedisIrisDevUI.dll --print-context-seed > artifacts/context-retriever/redis-iris-travel-context-seed.json


## Create or update the travel surface

The job compiles `Traveler`, `FlightReservation`, `HotelReservation`, and `TravelReceipt` from the
[shared sample source](../../samples/RedisIrisDevUI/TravelContext.cs), then passes the DataModel to `ctxctl`.
It creates one key per traveler with `tenant_id` and `traveler_id` access tags.

Artifacts appear under `artifacts/context-retriever`: the public model JSON and the private users credential bundle.
The configured surface is updated by default; duplicate names fail rather than selecting an arbitrary surface.


In [ ]:
#!pwsh
docker compose --profile context-retriever run --rm --build context-provisioner


## Import the synthetic records

The seeder serializes the sample POCO instances and calls the admin data-import endpoint once per entity:

```text
POST api/v1/context-surfaces/{surfaceId}/data
{ "entity": "Traveler", "records": [...],
  "options": { "on_conflict": "overwrite", "on_error": "fail_fast" } }
```

Expected counts: **2 travelers, 2 flights, 2 hotels, 3 receipts**. The job checks both `imported` and `failed`.
Imports are separate operations; a failure can leave earlier groups imported. Correct the issue and rerun to
reset the stable demo records. This is an administrative ingestion operation, not an agent tool call.


In [ ]:
#!pwsh
docker compose --profile context-retriever run --rm --build context-seeder


## Inspect only the public artifacts

The model declares fields, relationships, and ACL mappings; the seed contains synthetic records. Neither is a
credential bundle. Notebook 03 verifies actual ingestion by reading a known record over the scoped MCP connection.
For an existing fixture, locally exported JSON describes the expected fixture, not proof that remote data matches.


In [ ]:
#!pwsh
$model = Get-Content -Raw artifacts/context-retriever/redis-iris-travel-context-model.json | ConvertFrom-Json
$model.entities | ForEach-Object { "Entity: $($_.name) | key: $($_.redis_key_template)" }
$seed = Get-Content -Raw artifacts/context-retriever/redis-iris-travel-context-seed.json | ConvertFrom-Json
$seed.PSObject.Properties | ForEach-Object { "$($_.Name): $($_.Value.Count) synthetic records" }


## Keep the fixture for the series

Notebooks 03–05 dispose their connections but keep this shared surface and seed records. Notebook 05 separately
cleans up its synthetic RAM sessions and the LTM IDs it finds. Surface/key removal and backing-data deletion are
separate administrative lifecycle operations; do not delete a shared database or assume deleting a surface erases its data.


Next: open **03 — Context Retriever fundamentals** with a fresh **C#** kernel started in `notebooks/csharp`.
Notebook 00 changes its shell's working directory to the repository root; notebooks 03–05 use relative C# assembly paths.
Clear setup output before sharing an executed notebook.
